In [1]:
# === Environment setup ===
# Runs on Colab Pro, High-RAM CPU runtime
# Runtime > Change runtime type > CPU, High-RAM

# installation
!pip install -q scanpy anndata harmonypy leidenalg python-igraph

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import os

sc.settings.verbosity = 1
print("scanpy", sc.__version__)
print("anndata", ad.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 84.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sourc

/tmp/ipykernel_5572/4206305344.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy", sc.__version__)
/tmp/ipykernel_5572/4206305344.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata", ad.__version__)


In [3]:
# === Mount Drive + folder structure ===
from google.colab import drive
drive.mount('/content/drive')

# root folder
PROJECT_ROOT = '/content/drive/MyDrive/MLCB_team_project'
RAW_DIR       = os.path.join(PROJECT_ROOT, 'data', 'raw')        # whatever is downloaded from GEO
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'data', 'checkpoints') # AnnData .h5ad after every step

for d in (RAW_DIR, CHECKPOINT_DIR):
    os.makedirs(d, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data ->", RAW_DIR)
print("Checkpoints ->", CHECKPOINT_DIR)

Mounted at /content/drive
Project root: /content/drive/MyDrive/MLCB_team_project
Raw data -> /content/drive/MyDrive/MLCB_team_project/data/raw
Checkpoints -> /content/drive/MyDrive/MLCB_team_project/data/checkpoints


In [8]:

!pip install -q GEOparse
import GEOparse

# downloads SOFT metadata + list with supplementary files χωρίς μάντεμα URL
gse = GEOparse.get_GEO(geo="GSE213982", destdir=os.path.join(RAW_DIR, 'GSE213982'))
print("Supplementary files του Series:")
for url in gse.metadata.get('supplementary_file', []):
    print("  ", url)

# per sample (GSM)
print("\nPer sample (first 5 GSMs):")
for gsm_name, gsm in list(gse.gsms.items())[:5]:
    print(f"  {gsm_name}: {gsm.metadata.get('title', ['?'])[0]}")
    for url in gsm.metadata.get('supplementary_file', []):
        print("      ", url)

06-Jun-2026 07:28:17 DEBUG utils - Directory /content/drive/MyDrive/MLCB_team_project/data/raw/GSE213982 already exists. Skipping.
DEBUG:GEOparse:Directory /content/drive/MyDrive/MLCB_team_project/data/raw/GSE213982 already exists. Skipping.
06-Jun-2026 07:28:17 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
06-Jun-2026 07:28:17 INFO GEOparse - Parsing /content/drive/MyDrive/MLCB_team_project/data/raw/GSE213982/GSE213982_family.soft.gz: 
INFO:GEOparse:Parsing /content/drive/MyDrive/MLCB_team_project/data/raw/GSE213982/GSE213982_family.soft.gz: 
06-Jun-2026 07:28:17 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
06-Jun-2026 07:28:17 DEBUG GEOparse - SERIES: GSE213982
DEBUG:GEOparse:SERIES: GSE213982
06-Jun-2026 07:28:17 DEBUG GEOparse - PLATFORM: GPL24676
DEBUG:GEOparse:PLATFORM: GPL24676
06-Jun-2026 07:28:17 DEBUG GEOparse - PLATFORM: GPL28038
DEBUG:GEOparse:PLATFORM: GPL28038
06-Jun-2026 07:28:17 D

Supplementary files του Series:
   ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/GSE213982/suppl/GSE213982_combined_counts_matrix.mtx.gz
   ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/GSE213982/suppl/GSE213982_combined_counts_matrix_cells_columns.csv.gz
   ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/GSE213982/suppl/GSE213982_combined_counts_matrix_genes_rows.csv.gz

Per sample (first 5 GSMs):
  GSM6597355: F1
  GSM6597356: F10
  GSM6597357: F11
  GSM6597358: F12
  GSM6597359: F13


In [9]:
# === download the 3 supplementary files for GSE213982 + inspect structure ===
import os, gzip, subprocess
import pandas as pd
import scipy.io as sio

GSE = 'GSE213982'
dest_dir = os.path.join(RAW_DIR, GSE)
os.makedirs(dest_dir, exist_ok=True)

# Direct target-dir URLs (bypass parent dir to avoid the time-out that caused exit status 8)
base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/{GSE}/suppl/"
files = [
    f"{GSE}_combined_counts_matrix.mtx.gz",
    f"{GSE}_combined_counts_matrix_cells_columns.csv.gz",
    f"{GSE}_combined_counts_matrix_genes_rows.csv.gz",
]

for fn in files:
    out = os.path.join(dest_dir, fn)
    if not os.path.exists(out) or os.path.getsize(out) == 0:
        print(f"Downloading {fn} ...")
        # -c resume; check the return code explicitly this time
        r = subprocess.run(["wget", "-c", "-O", out, base + fn])
        if r.returncode != 0:
            print(f"  WARNING: wget returned {r.returncode} for {fn}")
    print(f"  {fn}: {os.path.getsize(out)/1e6:.1f} MB")

  GSE213982_combined_counts_matrix.mtx.gz: 1115.6 MB
  GSE213982_combined_counts_matrix_cells_columns.csv.gz: 1.1 MB
  GSE213982_combined_counts_matrix_genes_rows.csv.gz: 0.1 MB


In [10]:
# === inspect the metadata CSVs (this decides load_dataset) ===

cells_fn = os.path.join(dest_dir, f"{GSE}_combined_counts_matrix_cells_columns.csv.gz")
genes_fn = os.path.join(dest_dir, f"{GSE}_combined_counts_matrix_genes_rows.csv.gz")

# Read cell metadata — the critical file. Try with header first.
cells = pd.read_csv(cells_fn)
print("=== CELLS (columns of the matrix) ===")
print("shape:", cells.shape)
print("columns:", list(cells.columns))
print(cells.head(10).to_string())
print()

genes = pd.read_csv(genes_fn)
print("=== GENES (rows of the matrix) ===")
print("shape:", genes.shape)
print("columns:", list(genes.columns))
print(genes.head(5).to_string())

=== CELLS (columns of the matrix) ===
shape: (160711, 1)
columns: ['x']
                                    x
0      F1.AAACCCACACCTCTGT-1.Mic.Mic1
1      F1.AAACCCATCGGCATTA-1.ExN.ExN7
2  F1.AAACGAATCAGGGTAG-1.InN.InN8_Mix
3      F1.AAACGAATCGGAATGG-1.Oli.Oli3
4      F1.AAACGCTAGGACTTCT-1.Ast.Ast1
5     F1.AAACGCTGTGACTAAA-1.ExN.ExN17
6      F1.AAACGCTGTGCATGAG-1.Mic.Mic1
7      F1.AAACGCTTCACCACAA-1.Mic.Mic1
8  F1.AAACGCTTCTAAGCCA-1.ExN.ExN8_L24
9     F1.AAACGCTTCTTACGGA-1.ExN.ExN17

=== GENES (rows of the matrix) ===
shape: (36588, 1)
columns: ['x']
             x
0  MIR1302-2HG
1      FAM138A
2        OR4F5
3   AL627309.1
4   AL627309.3


In [11]:
# === check matrix orientation WITHOUT loading the whole thing ===
# Just read the MatrixMarket header line (dimensions), don't parse the body.
mtx_fn = os.path.join(dest_dir, f"{GSE}_combined_counts_matrix.mtx.gz")
with gzip.open(mtx_fn, 'rt') as f:
    for line in f:
        if line.startswith('%'):      # skip comment/header lines
            continue
        rows, cols, nnz = map(int, line.split())
        break
print(f"Matrix .mtx header: {rows} rows x {cols} cols, {nnz} non-zeros")
print(f"genes CSV rows: {genes.shape[0]}")
print(f"cells CSV rows: {cells.shape[0]}")
print()
print("Orientation check:")
print(f"  rows({rows}) == n_genes({genes.shape[0]})? {rows == genes.shape[0]}")
print(f"  cols({cols}) == n_cells({cells.shape[0]})? {cols == cells.shape[0]}")

Matrix .mtx header: 36588 rows x 160711 cols, 360887743 non-zeros
genes CSV rows: 36588
cells CSV rows: 160711

Orientation check:
  rows(36588) == n_genes(36588)? True
  cols(160711) == n_cells(160711)? True


In [12]:
# === Εxtract per-donor condition (MDD/Control) from GSM metadata ===
# The cell-metadata column 'x' encodes donor (F1, F2...) but NOT diagnosis.
# Diagnosis lives in each GSM's characteristics. We map donor_label -> condition.

import GEOparse, os, re

GSE = 'GSE213982'
dest_dir = os.path.join(RAW_DIR, GSE)

# Reuse the already-downloaded SOFT file (no re-download)
gse = GEOparse.get_GEO(geo=GSE, destdir=dest_dir, silent=True)

# Inspect what characteristics each GSM carries — print the first one fully
first_gsm = list(gse.gsms.values())[0]
print("=== Full metadata keys of first GSM ===")
for k, v in first_gsm.metadata.items():
    print(f"  {k}: {v}")

=== Full metadata keys of first GSM ===
  title: ['F1']
  geo_accession: ['GSM6597355']
  status: ['Public on Apr 24 2023']
  submission_date: ['Sep 22 2022']
  last_update_date: ['Apr 24 2023']
  type: ['SRA']
  channel_count: ['1']
  source_name_ch1: ['Brain BA9 dlPFC']
  organism_ch1: ['Homo sapiens']
  taxid_ch1: ['9606']
  characteristics_ch1: ['tissue: Brain BA9 dlPFC', 'Sex: Female', 'group: Case']
  molecule_ch1: ['polyA RNA']
  extract_protocol_ch1: ['Chromium Single Cell 3’ Reagent Kits v2 and v3.1']
  description: ['Batch: 8F, Chemistry:  v3']
  data_processing: ['10X Genomics Cellranger v3.1.0 mkfastq', '10X Genomics Cellranger v5.0.1 count', 'Cell filtering as described in publication', 'Clustering and cell type annotation: Unsupervised clustering of nuclei was performed with the Seurat algorithm, after using highly variable genes for PCA and correcting for batch effects with the Harmony R package. Clusters were annotated to cell types based on the enrichment of expression

In [13]:
# === Βuild donor -> condition map ===

donor_condition = {}
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata['title'][0]            # e.g. 'F1', 'F10' -> this is our donor_id
    chars = gsm.metadata.get('characteristics_ch1', [])
    # Look for a diagnosis/phenotype/group field among the characteristics
    cond = None
    for c in chars:
        c_low = c.lower()
        if any(k in c_low for k in ['diagnosis', 'phenotype', 'group', 'disease', 'condition', 'status']):
            val = c.split(':', 1)[1].strip().lower()
            # Normalise to two labels
            if any(w in val for w in ['mdd', 'case', 'depress', 'suicide']):
                cond = 'MDD'
            elif any(w in val for w in ['control', 'ctrl', 'healthy', 'neurotypical']):
                cond = 'Control'
            else:
                cond = val  # keep raw so we can see anything unexpected
            break
    donor_condition[title] = cond

# Sanity check: how many MDD vs Control, any None (unmapped)?
import pandas as pd
s = pd.Series(donor_condition)
print("Per-donor condition counts:")
print(s.value_counts(dropna=False).to_string())
print(f"\nTotal donors: {len(donor_condition)}")
print("Unmapped (None) donors:", [d for d, c in donor_condition.items() if c is None])
print("\nFull map:")
print(s.sort_index().to_string())

Per-donor condition counts:
MDD        20
Control    18

Total donors: 38
Unmapped (None) donors: []

Full map:
F1         MDD
F10    Control
F11        MDD
F12        MDD
F13    Control
F14        MDD
F15        MDD
F16        MDD
F17        MDD
F18        MDD
F19        MDD
F2         MDD
F20        MDD
F21    Control
F22    Control
F23    Control
F24    Control
F25        MDD
F26    Control
F27        MDD
F28        MDD
F29    Control
F3         MDD
F30    Control
F31    Control
F32    Control
F33    Control
F34    Control
F35    Control
F36    Control
F37    Control
F38    Control
F4         MDD
F5         MDD
F6         MDD
F7     Control
F8         MDD
F9         MDD


In [14]:
# === Verify the combined matrix already contains BOTH sexes ===
# Re-read the cells metadata and parse the donor token (first field before '.')

cells_fn = os.path.join(RAW_DIR, 'GSE213982',
                        'GSE213982_combined_counts_matrix_cells_columns.csv.gz')
cells = pd.read_csv(cells_fn)

# 'x' looks like: F1.AAACCCACACCTCTGT-1.Mic.Mic1
donor_tokens = cells['x'].str.split('.', n=1).str[0]
unique_donors = sorted(donor_tokens.unique())

print(f"Total cells: {len(cells)}")
print(f"Unique donors: {len(unique_donors)}")
print(f"Donors: {unique_donors}")
print()

# Split by leading letter to see sexes
female_donors = [d for d in unique_donors if d.startswith('F')]
male_donors   = [d for d in unique_donors if d.startswith('M')]
print(f"Female donors (F*): {len(female_donors)} -> {female_donors}")
print(f"Male donors   (M*): {len(male_donors)} -> {male_donors}")
print()
print("Cells per sex:")
print(donor_tokens.str[0].value_counts().to_string())

Total cells: 160711
Unique donors: 72
Donors: ['F1', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F2', 'F20', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28', 'F29', 'F3', 'F30', 'F31', 'F32', 'F33', 'F34', 'F35', 'F36', 'F37', 'F38', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9', 'M1', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M19', 'M2', 'M20', 'M21', 'M22', 'M23', 'M24', 'M24_2', 'M26', 'M27', 'M28', 'M29', 'M3', 'M30', 'M31', 'M32', 'M33', 'M34', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']

Female donors (F*): 38 -> ['F1', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F2', 'F20', 'F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28', 'F29', 'F3', 'F30', 'F31', 'F32', 'F33', 'F34', 'F35', 'F36', 'F37', 'F38', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9']
Male donors   (M*): 34 -> ['M1', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M19', 'M2', 'M20', 'M21', 'M22', 'M23', 'M24', 'M24_2', 'M26', 'M27', 'M28', 'M29',

In [15]:
# === Check whether the GSE213982 SOFT covers male donors too ===
# We already built donor_condition from GSE213982's GSMs. Did it include M* donors?

mapped_donors = set(donor_condition.keys())
matrix_donors = set(unique_donors)   # the 72 from the matrix

print("Donors in matrix but NOT in condition map:")
missing = sorted(matrix_donors - mapped_donors)
print(f"  {len(missing)}: {missing}")
print()
print("Donors in condition map but NOT in matrix:")
print(f"  {sorted(mapped_donors - matrix_donors)}")

Donors in matrix but NOT in condition map:
  34: ['M1', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M19', 'M2', 'M20', 'M21', 'M22', 'M23', 'M24', 'M24_2', 'M26', 'M27', 'M28', 'M29', 'M3', 'M30', 'M31', 'M32', 'M33', 'M34', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']

Donors in condition map but NOT in matrix:
  []


In [16]:
# === Pull male donor conditions from GSE144136 SOFT (metadata only, ~KB) ===
gse_male = GEOparse.get_GEO(geo='GSE144136',
                            destdir=os.path.join(RAW_DIR, 'GSE144136'), silent=True)

# Inspect one male GSM to find where diagnosis lives (may differ from female study)
first_male = list(gse_male.gsms.values())[0]
print("=== First GSE144136 GSM metadata ===")
print("title:", first_male.metadata['title'])
print("characteristics_ch1:", first_male.metadata.get('characteristics_ch1'))
print("source_name_ch1:", first_male.metadata.get('source_name_ch1'))

=== First GSE144136 GSM metadata ===
title: ['1: Post-mortem dosolateral prefrontal cortex']
characteristics_ch1: ['group: Major Depressive Disorder (MDD)', 'Sex: Male', 'ethnicity: Caucasian', 'tissue: Brain']
source_name_ch1: ['Post-mortem dosolateral prefrontal cortex']


In [17]:
# === Dump ALL GSE144136 GSMs to understand title -> donor mapping ===
print(f"Total GSE144136 GSMs: {len(gse_male.gsms)}\n")

rows = []
for gsm_name, gsm in gse_male.gsms.items():
    title = gsm.metadata['title'][0]
    chars = gsm.metadata.get('characteristics_ch1', [])
    cond_raw = next((c for c in chars if c.lower().startswith('group')), None)
    sex_raw  = next((c for c in chars if c.lower().startswith('sex')), None)
    rows.append({'gsm': gsm_name, 'title': title,
                 'group': cond_raw, 'sex': sex_raw})

male_meta = pd.DataFrame(rows)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(male_meta.to_string())

Total GSE144136 GSMs: 34

           gsm                                          title                                   group        sex
0   GSM4281974   1: Post-mortem dosolateral prefrontal cortex  group: Major Depressive Disorder (MDD)  Sex: Male
1   GSM4281975   2: Post-mortem dosolateral prefrontal cortex                          group: Control  Sex: Male
2   GSM4281976   3: Post-mortem dosolateral prefrontal cortex                          group: Control  Sex: Male
3   GSM4281977   4: Post-mortem dosolateral prefrontal cortex  group: Major Depressive Disorder (MDD)  Sex: Male
4   GSM4281978   5: Post-mortem dosolateral prefrontal cortex  group: Major Depressive Disorder (MDD)  Sex: Male
5   GSM4281979   6: Post-mortem dosolateral prefrontal cortex  group: Major Depressive Disorder (MDD)  Sex: Male
6   GSM4281980   7: Post-mortem dosolateral prefrontal cortex                          group: Control  Sex: Male
7   GSM4281981   8: Post-mortem dosolateral prefrontal cortex  group: 

In [18]:
# === Build male donor->condition from GSE144136 titles, then reconcile with matrix ===
import re

male_condition_by_num = {}
for gsm_name, gsm in gse_male.gsms.items():
    title = gsm.metadata['title'][0]          # e.g. '1: Post-mortem...'
    num = int(title.split(':', 1)[0].strip()) # donor number 1..34
    chars = gsm.metadata.get('characteristics_ch1', [])
    grp = next((c for c in chars if c.lower().startswith('group')), '')
    cond = 'MDD' if 'mdd' in grp.lower() or 'depress' in grp.lower() else 'Control'
    male_condition_by_num[num] = cond

print("Condition by donor number (1..34):")
for n in sorted(male_condition_by_num):
    print(f"  {n}: {male_condition_by_num[n]}")

import pandas as pd
counts = pd.Series(male_condition_by_num).value_counts()
print(f"\nMDD: {counts.get('MDD',0)}, Control: {counts.get('Control',0)}")

# Now the actual male donor labels present in the matrix:
matrix_males = sorted([d for d in unique_donors if d.startswith('M')],
                      key=lambda s: (int(re.match(r'M(\d+)', s).group(1)), s))
print(f"\nMale donor labels in matrix ({len(matrix_males)}):")
print(matrix_males)

# Where do 24 / 24_2 / 25 / 26 sit? Inspect the join point.
print("\nMatrix males around the 24/25 boundary:",
      [d for d in matrix_males if d in ('M23','M24','M24_2','M25','M26','M27')])
print("Title numbers around there: 23,24,25,26,27 ->",
      {n: male_condition_by_num[n] for n in (23,24,25,26,27)})

Condition by donor number (1..34):
  1: MDD
  2: Control
  3: Control
  4: MDD
  5: MDD
  6: MDD
  7: Control
  8: MDD
  9: Control
  10: MDD
  11: MDD
  12: Control
  13: Control
  14: MDD
  15: Control
  16: Control
  17: MDD
  18: MDD
  19: Control
  20: Control
  21: Control
  22: Control
  23: MDD
  24: Control
  25: Control
  26: MDD
  27: Control
  28: MDD
  29: Control
  30: MDD
  31: Control
  32: MDD
  33: MDD
  34: MDD

MDD: 17, Control: 17

Male donor labels in matrix (34):
['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M19', 'M20', 'M21', 'M22', 'M23', 'M24', 'M24_2', 'M26', 'M27', 'M28', 'M29', 'M30', 'M31', 'M32', 'M33', 'M34']

Matrix males around the 24/25 boundary: ['M23', 'M24', 'M24_2', 'M26', 'M27']
Title numbers around there: 23,24,25,26,27 -> {23: 'MDD', 24: 'Control', 25: 'Control', 26: 'MDD', 27: 'Control'}
